# Personal AI Assistant with Long-Term Memory

Build an AI assistant that remembers everything you've told it across conversations. Unlike typical chatbots that forget after each session, this assistant uses Engram to recall past discussions.

**Use case**: A personal assistant that knows your preferences, ongoing projects, and life context.

In [ ]:
# !pip install engram-search

In [ ]:
from engram.backends.faiss_backend import FaissBackend
from engram.backends.base import Document
from engram.retrieval.embedder import Embedder
from engram.retrieval.sparse import BM25
from engram.retrieval.pipeline import reciprocal_rank_fusion
from engram.ingestion.parser import session_to_documents, is_assistant_reference

backend = FaissBackend(path="./assistant_memory", dimension=1024)
embedder = Embedder("bge-large")

print("Personal assistant memory initialized")

## 1. Build Up Memory Over Time

Simulate weeks of conversations with your AI assistant.

In [ ]:
past_conversations = [
    {
        "id": "week1_mon",
        "timestamp": "2024-03-04",
        "turns": [
            {"role": "user", "content": "I just started a new job as a senior backend engineer at a fintech startup called PayFlow."},
            {"role": "assistant", "content": "Congratulations on the new role! What tech stack does PayFlow use?"},
            {"role": "user", "content": "We use Go for microservices, PostgreSQL for the database, and Kafka for event streaming. I'm mainly working on the payments processing pipeline."},
            {"role": "assistant", "content": "Great stack! Go is excellent for high-throughput payment systems. Let me know if you need help with any of those technologies."}
        ]
    },
    {
        "id": "week1_wed",
        "timestamp": "2024-03-06",
        "turns": [
            {"role": "user", "content": "My wife Emma and I are planning a trip to Japan in May. We want to visit Tokyo and Kyoto."},
            {"role": "assistant", "content": "Japan in May is beautiful! The weather is mild and it's past the cherry blossom crowds."},
            {"role": "user", "content": "Emma is vegetarian so we need to find good restaurants. Also, she loves traditional tea ceremonies."},
            {"role": "assistant", "content": "Kyoto has excellent vegetarian options, especially at temple restaurants (shojin ryori). For tea ceremonies, I'd recommend the Urasenke Chado Research Center in Kyoto."}
        ]
    },
    {
        "id": "week2_tue",
        "timestamp": "2024-03-12",
        "turns": [
            {"role": "user", "content": "I've been having issues with our Kafka consumer lag at work. Messages are piling up during peak hours."},
            {"role": "assistant", "content": "Consumer lag usually means your processing can't keep up. Have you looked at partition count and consumer group scaling?"},
            {"role": "user", "content": "We have 12 partitions and 4 consumers. I think we need to increase the consumer count to 12."},
            {"role": "assistant", "content": "Exactly — matching consumers to partitions is the right approach. Each consumer gets exclusive partitions, so 12:12 is ideal. Also consider batching if individual message processing is slow."}
        ]
    },
    {
        "id": "week3_fri",
        "timestamp": "2024-03-22",
        "turns": [
            {"role": "user", "content": "I'm training for a half marathon in June. My current best 5K is 24 minutes."},
            {"role": "assistant", "content": "A 24-minute 5K is a solid base! For half marathon prep, you'll want to build up your long runs gradually."},
            {"role": "user", "content": "I usually run 3 times a week. Tuesday, Thursday, and a long run on Sunday."},
            {"role": "assistant", "content": "Great schedule. Add one more easy run if possible. For your long runs, increase distance by no more than 10% per week to avoid injury."}
        ]
    },
    {
        "id": "week4_mon",
        "timestamp": "2024-03-25",
        "turns": [
            {"role": "user", "content": "Big news — I got promoted to tech lead at PayFlow! I'll be managing a team of 6 engineers now."},
            {"role": "assistant", "content": "That's amazing, congratulations! Going from IC to tech lead is a big transition."},
            {"role": "user", "content": "Yeah, I'm a bit nervous about the people management side. I've always been more of a code person."},
            {"role": "assistant", "content": "That's completely natural. I'd recommend 'The Manager's Path' by Camille Fournier — it's written specifically for engineers transitioning to leadership."}
        ]
    }
]

# Ingest all conversations
all_docs = []
for conv in past_conversations:
    parsed = session_to_documents(
        session=conv["turns"],
        session_id=conv["id"],
        timestamp=conv["timestamp"],
        include_assistant=True,
    )
    all_docs.extend(parsed)

texts = [d["text"] for d in all_docs]
embeddings = embedder.encode_documents(texts)
documents = [
    Document(id=d["id"], text=d["text"], embedding=embeddings[i].tolist(), metadata=d["metadata"])
    for i, d in enumerate(all_docs)
]
backend.add(documents)

print(f"Loaded {len(past_conversations)} weeks of conversations ({len(documents)} documents)")

## 2. The Assistant Remembers

Now when you ask a question, the assistant has full context across all past conversations.

In [ ]:
def recall(query, top_k=3):
    """Retrieve relevant memories with hybrid search."""
    query_vec = embedder.encode_query(query)
    dense_results = backend.query(embedding=query_vec.tolist(), top_k=top_k * 3)
    dense_ranking = [(d.id, d.score) for d in dense_results]
    
    bm25 = BM25()
    all_texts = [d.text for d in documents]
    all_ids = [d.id for d in documents]
    bm25_scores = bm25.score_query_against_docs(query, all_texts)
    sparse_ranking = sorted(zip(all_ids, bm25_scores), key=lambda x: x[1], reverse=True)
    
    fused = reciprocal_rank_fusion(dense_ranking, sparse_ranking)
    
    id_to_doc = {d.id: d for d in documents}
    print(f'Recalling: "{query}"\n{"─" * 60}')
    for doc_id, score in fused[:top_k]:
        if doc_id in id_to_doc:
            doc = id_to_doc[doc_id]
            meta = doc.metadata or {}
            print(f"  [{meta.get('session_id', '?')}] {meta.get('timestamp', '')} (score: {score:.4f})")
            print(f"  {doc.text[:150]}...")
            print()

In [ ]:
# The assistant knows where you work
recall("Where do I work and what's my role?")

In [ ]:
# It remembers your personal life too
recall("What trip am I planning?")

In [ ]:
# It can cross-reference topics
recall("What technical issues have I been dealing with at work?")

In [ ]:
# It knows your spouse's name and preferences
recall("What does Emma like?")

In [ ]:
# Fitness goals
recall("What's my running schedule?")

## 3. Detecting Assistant References

Engram can detect when users ask about what the AI previously said — "What did you recommend?" — and search assistant turn documents specifically.

In [ ]:
test_questions = [
    "What book did you recommend for management?",
    "You mentioned something about Kafka batching",
    "What restaurant did you suggest in Kyoto?",
    "Tell me about Go",  # Not an assistant reference
]

for q in test_questions:
    is_ref = is_assistant_reference(q)
    print(f"  {'[ASST-REF]' if is_ref else '[GENERAL] '} {q}")

## 4. Building a Memory-Augmented Prompt

Here's how you'd build a prompt for any LLM (GPT-4, Claude, Llama, etc.) with memory context.

In [ ]:
def assistant_respond(user_message, top_k=3):
    """Build a memory-augmented prompt for the assistant."""
    # Retrieve relevant memories
    query_vec = embedder.encode_query(user_message)
    results = backend.query(embedding=query_vec.tolist(), top_k=top_k)
    
    # Build context from memories
    memory_context = ""
    for doc in results:
        meta = doc.metadata or {}
        memory_context += f"\n[{meta.get('timestamp', 'unknown date')}]: {doc.text[:200]}\n"
    
    prompt = f"""You are a personal AI assistant with long-term memory.
You remember past conversations with this user.

RELEVANT MEMORIES:
{memory_context}

USER: {user_message}

Respond naturally, referencing past context when helpful. Don't list all memories — just use what's relevant."""
    
    print("Generated prompt for LLM:")
    print("=" * 60)
    print(prompt)
    print("=" * 60)
    print("\n(Pass this to any LLM API — OpenAI, Anthropic, local Llama, etc.)")

# Example: user follows up on a past topic
assistant_respond("How's my half marathon training going? Any tips for this week?")

## 5. Continuous Memory

After each conversation, ingest the new session to keep memory growing.

In [ ]:
# New conversation happens today
new_session = [
    {"role": "user", "content": "I booked the Japan flights! Tokyo from May 10-17, then Kyoto May 17-22."},
    {"role": "assistant", "content": "Exciting! That's a great split — a week in Tokyo and 5 days in Kyoto."},
]

# Ingest the new session
new_docs = session_to_documents(
    session=new_session,
    session_id="week5_tue",
    timestamp="2024-04-02",
    include_assistant=True,
)

new_texts = [d["text"] for d in new_docs]
new_embeddings = embedder.encode_documents(new_texts)
new_documents = [
    Document(id=d["id"], text=d["text"], embedding=new_embeddings[i].tolist(), metadata=d["metadata"])
    for i, d in enumerate(new_docs)
]
backend.add(new_documents)

print(f"Memory updated. Total documents: {backend.count()}")

# Now the assistant knows the exact dates
query_vec = embedder.encode_query("When is my Japan trip?")
results = backend.query(embedding=query_vec.tolist(), top_k=2)
for r in results:
    print(f"  [{r.metadata.get('session_id')}] {r.text[:100]}...")

## Cleanup

In [ ]:
import shutil
shutil.rmtree("./assistant_memory", ignore_errors=True)
print("Assistant memory cleaned up.")